# RAG Assessment (RAGAS): Practical Demo

A RAG system can fail in different ways: the retriever returns noise, the retriever misses information, the LLM hallucinates, or the answer drifts away from the question. **RAGAS** measures these failures with four metrics, so we know **which part** of the RAG system to fix.

| Metric | Question it answers | Part of RAG it checks |
|---|---|---|
| **Faithfulness** | Does the answer contain only claims supported by the retrieved chunks? | Generation (hallucination) |
| **Answer relevance** | Does the answer address the question that was asked? | Generation (off-topic) |
| **Context precision** | Out of the chunks retrieved, how many were useful? | Retrieval (noise) |
| **Context recall** | Out of what we needed, how much was retrieved? | Retrieval (missing information) |

All four metrics use a **judge LLM** (*LLM as Judge*): a bigger, more powerful model than the one inside the RAG system. Context metrics also need **ground truth**: questions with pre-written correct answers.

**What we will do**
1. Setup (RAG LLM + bigger judge LLM)
2. Build a real RAG system on our text
3. Write the four metrics
4. See one metric in detail
5. Break the RAG system in five different ways and watch the scores
6. Summary and diagnosis

## 1. Setup

We use two LLMs, both from Groq:
- `openai/gpt-oss-20b`: the LLM **inside** the RAG system (generates answers)
- `openai/gpt-oss-120b`: the bigger **judge** that grades the answers

In [1]:
# Bring the api_key from secrets into a variable
from google.colab import userdata

my_api_key = userdata.get('agentic_ai')

In [2]:
pip install langchain-groq langchain-huggingface -q # q for silent installation

In [3]:
from langchain_groq import ChatGroq

# The LLM inside the RAG system
llm = ChatGroq(
    model = 'openai/gpt-oss-20b',
    temperature = 0, # Creativity level of the model
    api_key = my_api_key,
    max_retries = 6  # retry automatically if Groq's rate limit is reached
)

# The bigger judge LLM
judge_llm = ChatGroq(
    model = 'openai/gpt-oss-120b',
    temperature = 0,
    api_key = my_api_key,
    max_retries = 6
)

## 2. A real RAG system on our text

### 2.1 Text and chunks

One chunk per sentence (6 chunks).

In [4]:
import re

text = """Artificial Intelligence (AI) is transforming healthcare by improving disease diagnosis, patient monitoring, and treatment planning. Machine learning models analyze medical records, laboratory reports, and medical images to identify patterns and predict potential health risks.

Deep learning techniques, such as Convolutional Neural Networks (CNNs), are widely used for medical image classification. Natural Language Processing (NLP) helps extract meaningful information from clinical notes and electronic health records.

However, AI adoption in healthcare faces challenges, including data privacy, model bias, high implementation costs, and the need for reliable predictions. Combining AI technologies with human expertise can improve healthcare efficiency and patient outcomes."""

# Split after . ! ? followed by whitespace
chunks = re.split(r'(?<=[.!?])\s+', text)

for i, chunk in enumerate(chunks):
    print(i, '|', chunk)

0 | Artificial Intelligence (AI) is transforming healthcare by improving disease diagnosis, patient monitoring, and treatment planning.
1 | Machine learning models analyze medical records, laboratory reports, and medical images to identify patterns and predict potential health risks.
2 | Deep learning techniques, such as Convolutional Neural Networks (CNNs), are widely used for medical image classification.
3 | Natural Language Processing (NLP) helps extract meaningful information from clinical notes and electronic health records.
4 | However, AI adoption in healthcare faces challenges, including data privacy, model bias, high implementation costs, and the need for reliable predictions.
5 | Combining AI technologies with human expertise can improve healthcare efficiency and patient outcomes.


### 2.2 Retriever

Chunks are embedded and stored in a vector store (in memory).

In [5]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
store = InMemoryVectorStore.from_texts(chunks, embedding=embeddings)

### 2.3 Ground truth

Questions with **pre-written correct answers**, taken from our text. The third question needs **two chunks** to be answered completely.

In [6]:
dataset = [
    {'question': 'What are CNNs used for?',
     'ground_truth': 'CNNs are widely used for medical image classification.'},
    {'question': 'What challenges does AI adoption face in healthcare?',
     'ground_truth': 'AI adoption in healthcare faces challenges including data privacy, model bias, high implementation costs, and the need for reliable predictions.'},
    {'question': 'Which techniques are used for medical images and for clinical notes?',
     'ground_truth': 'Convolutional Neural Networks are used for medical image classification, and Natural Language Processing is used to extract information from clinical notes.'},
]

### 2.4 The RAG pipeline: retrieve → augment → generate

`k` is the number of chunks retrieved. `prompt` tells the LLM how to answer. We will change both later to break the system on purpose.

In [7]:
# Prompt 1: strict. The LLM may only use the retrieved chunks
strict_prompt = """Answer the question using ONLY the context below.
If the answer is not in the context, say "I don't know".

Context:
{context}

Question: {question}"""

In [8]:
def rag(question, k, prompt):
    # Retrieval: the k most similar chunks
    retrieved = [doc.page_content for doc in store.similarity_search(question, k=k)]

    # Augmentation: combine the chunks with the question
    context = '\n'.join(retrieved)

    # Generation: the LLM answers
    answer = llm.invoke(prompt.format(context=context, question=question)).content
    return retrieved, answer

## 3. The four metrics

The judge LLM answers in a very simple format: **one line per item**, like `item | yes` or `item | no`. A small Python function reads these lines.

In [9]:
import numpy as np

# Read the judge's answer: one line per item, in the format  item | yes   or   item | no
def parse_verdicts(output):
    results = []
    for line in output.splitlines():
        line = line.strip().strip('|').strip()   # remove table borders if the judge adds them
        if '|' in line:
            item, verdict = line.rsplit('|', 1)
            results.append((item.strip(), verdict.strip().lower().startswith('yes')))
    return results

# Score = share of items with verdict yes (nan if the judge found no items)
def share_yes(results):
    return np.mean([verdict for _, verdict in results]) if results else np.nan

### 3.1 Faithfulness

The judge breaks the answer into individual **claims** (at most 8, to keep its answer short) and checks each claim against the retrieved chunks.

`score = supported claims / all claims` (1.0 means no hallucination)

In [10]:
def faithfulness(answer, retrieved):
    prompt = f"""Break the ANSWER into short individual claims (at most 8).
Write one line per claim in exactly this format:
claim | yes    (if the CONTEXT supports the claim)
claim | no     (if the CONTEXT does not support the claim)
Write nothing else.

CONTEXT:
{chr(10).join(retrieved)}

ANSWER:
{answer}"""
    results = parse_verdicts(judge_llm.invoke(prompt).content)
    return share_yes(results), results

### 3.2 Answer relevance

The judge writes **3 hypothetical questions** that the answer would be a good response to. We then measure how **similar** they are to the original question (cosine similarity of embeddings).

High similarity means the answer is on-topic. Low similarity means it drifted.

In [11]:
def answer_relevance(question, answer):
    prompt = f"""Write 3 different questions that the following ANSWER would be a good response to.
Write one question per line, without numbering. Write nothing else.

ANSWER:
{answer}"""
    output = judge_llm.invoke(prompt).content
    generated = [re.sub(r'^[\d\.\-\*\)\s]+', '', line).strip() for line in output.splitlines() if line.strip()]

    # Cosine similarity between each generated question and the original question
    original_vec = np.array(embeddings.embed_query(question))
    sims = []
    for vec in embeddings.embed_documents(generated):
        vec = np.array(vec)
        sims.append(np.dot(vec, original_vec) / (np.linalg.norm(vec) * np.linalg.norm(original_vec)))
    return np.mean(sims), generated

### 3.3 Context precision

The judge checks every retrieved chunk: is it **useful** for answering the question? (It sees the ground truth to know what the right answer is.)

`score = useful chunks / retrieved chunks` (40% means 60% of the retrieved chunks were noise)

In [12]:
def context_precision(question, ground_truth, retrieved):
    numbered = '\n'.join(f'{i + 1}. {chunk}' for i, chunk in enumerate(retrieved))
    prompt = f"""For each numbered CONTEXT CHUNK, decide if it is useful for giving the correct answer to the QUESTION.
Write one line per chunk in exactly this format:
chunk number | yes    (useful)
chunk number | no     (not useful)
Write nothing else.

QUESTION: {question}
CORRECT ANSWER (ground truth): {ground_truth}

CONTEXT CHUNKS:
{numbered}"""
    results = parse_verdicts(judge_llm.invoke(prompt).content)
    return share_yes(results), results

### 3.4 Context recall

The judge splits the **ground truth** into statements and checks if each statement can be **found** in the retrieved chunks.

`score = statements found / all statements` (40% means 60% of the needed information was missed)

In [13]:
def context_recall(ground_truth, retrieved):
    prompt = f"""Break the GROUND TRUTH into short individual statements (at most 8).
Write one line per statement in exactly this format:
statement | yes    (if it can be found in the CONTEXT)
statement | no     (if it cannot be found in the CONTEXT)
Write nothing else.

CONTEXT:
{chr(10).join(retrieved)}

GROUND TRUTH:
{ground_truth}"""
    results = parse_verdicts(judge_llm.invoke(prompt).content)
    return share_yes(results), results

### 3.5 All four metrics together

In [14]:
def evaluate(question, ground_truth, retrieved, answer):
    return {
        'faithfulness': faithfulness(answer, retrieved)[0],
        'answer_relevance': answer_relevance(question, answer)[0],
        'context_precision': context_precision(question, ground_truth, retrieved)[0],
        'context_recall': context_recall(ground_truth, retrieved)[0],
    }

## 4. One metric in detail

Before the full experiment, let us look inside **faithfulness**. We use a prompt that lets the LLM add facts from its own knowledge, so some claims should not be supported by our chunks.

In [15]:
# Prompt 2: the LLM may add its own knowledge (this causes hallucination)
hallucinate_prompt = """Answer the question using the context below.
Then add extra helpful details and examples from your own general knowledge.

Context:
{context}

Question: {question}"""

In [16]:
question = dataset[0]['question']
retrieved, answer = rag(question, k=2, prompt=hallucinate_prompt)

score, claims = faithfulness(answer, retrieved)

print('Question:', question)
print('Answer:', answer)
print()
print(f'Faithfulness = {score:.2f}')
for claim, verdict in claims:
    print('  supported' if verdict else '  NOT supported', '|', claim)

Question: What are CNNs used for?
Answer: **Short answer:**  
Convolutional Neural Networks (CNNs) are primarily used to analyze and interpret visual data. In medicine they are the go‑to tool for **classifying, detecting, and segmenting medical images**—for example, spotting tumors in MRIs, diagnosing pneumonia from chest X‑rays, or grading diabetic retinopathy from retinal photographs.

---

## 1. Core medical‑imaging tasks

| Task | What CNNs do | Typical examples |
|------|--------------|------------------|
| **Image classification** | Assign a single label (e.g., “normal” vs. “pneumonia”) | Chest X‑ray pneumonia detection, skin‑lesion malignancy classification |
| **Object detection** | Locate and label multiple objects in one image | Counting cells in histopathology slides, locating lung nodules in CT scans |
| **Semantic segmentation** | Assign a class to every pixel (e.g., tumor vs. healthy tissue) | Brain tumor segmentation in MRI, organ delineation in CT |
| **Instance segment

## 5. Break the RAG system in five ways

We run the same 3 questions in five scenarios. Each scenario causes **one** failure on purpose.

| Scenario | How we create it | Which metric should drop |
|---|---|---|
| Healthy RAG | k = 2, strict prompt | none |
| Low recall | k = 1: too few chunks retrieved | context recall |
| Low precision | k = 6: all chunks retrieved, mostly noise | context precision |
| Hallucination | prompt lets the LLM add its own knowledge | faithfulness |
| Off-topic answer | prompt asks for a general overview instead of an answer | answer relevance |

The expected pattern is only our prediction. The table below shows what really happens.

In [17]:
# Prompt 3: the LLM does not answer the question, it describes the general topic instead
overview_prompt = """Do not answer the question directly.
Instead, give a short general overview of the topic area of the question, using the context below.

Context:
{context}

Question: {question}"""

In [18]:
# scenario name -> (number of chunks retrieved, prompt)
scenarios = {
    'Healthy RAG':      (2, strict_prompt),
    'Low recall':       (1, strict_prompt),
    'Low precision':    (6, strict_prompt),
    'Hallucination':    (2, hallucinate_prompt),
    'Off-topic answer': (2, overview_prompt),
}

This makes about 75 LLM calls, so it can take a few minutes.

In [19]:
import pandas as pd

rows = []
for name, (k, prompt) in scenarios.items():
    print('Running:', name)
    for number, item in enumerate(dataset, start=1):
        retrieved, answer = rag(item['question'], k, prompt)
        scores = evaluate(item['question'], item['ground_truth'], retrieved, answer)
        rows.append({'scenario': name, 'question': f'Q{number}', **scores})

results = pd.DataFrame(rows).round(2)

Running: Healthy RAG
Running: Low recall
Running: Low precision
Running: Hallucination
Running: Off-topic answer


### 5.1 Scores for every question

The four metrics for each question (Q1, Q2, Q3) in each scenario (1.0 = perfect).

| Label | Question |
|---|---|
| Q1 | What are CNNs used for? |
| Q2 | What challenges does AI adoption face in healthcare? |
| Q3 | Which techniques are used for medical images and for clinical notes? |

In [20]:
results.set_index(['scenario', 'question'])

faithfulness  answer_relevance  context_precision  \
scenario         question                                                      
Healthy RAG      Q1                1.00              0.68               0.50   
                 Q2                1.00              0.80               0.50   
                 Q3                0.62              0.51               0.50   
Low recall       Q1                1.00              0.68               1.00   
                 Q2                1.00              0.78               1.00   
                 Q3                0.50              0.75               1.00   
Low precision    Q1                1.00              0.68               0.17   
                 Q2                1.00              0.80               0.17   
                 Q3                1.00              0.71               0.33   
Hallucination    Q1                0.25              0.54               0.50   
                 Q2                1.00              0.76               0.50   
                 Q3                0.25              0.65               0.50   
Off-topic answer Q1                0.38              0.31               0.50   
                 Q2                0.50              0.69               0.50   
                 Q3                0.12              0.56               0.50   

                           context_recall  
scenario         question                  
Healthy RAG      Q1                   1.0  
                 Q2                   1.0  
                 Q3                   0.5  
Low recall       Q1                   1.0  
                 Q2                   1.0  
                 Q3                   0.5  
Low precision    Q1                   1.0  
                 Q2                   1.0  
                 Q3                   1.0  
Hallucination    Q1                   1.0  
                 Q2                   1.0  
                 Q3                   0.5  
Off-topic answer Q1                   1.0  
                 Q2                   1.0  
                 Q3                   0.5

### 5.2 Average of the 3 questions for each scenario

In [21]:
results.groupby('scenario', sort=False).mean(numeric_only=True).round(2)

,faithfulness,answer_relevance,context_precision,context_recall
scenario,,,,
Healthy RAG,0.87,0.66,0.50,0.83
Low recall,0.83,0.74,1.00,0.83
Low precision,1.00,0.73,0.22,1.00
Hallucination,0.50,0.65,0.50,0.83
Off-topic answer,0.33,0.52,0.50,0.83


## 6. Summary

### How to read the table (diagnosis from our notes)
- **High faithfulness + low context recall** → retrieval is the bottleneck, the LLM is fine.
- **Low faithfulness + high context recall** → the LLM hallucinates, retrieval is fine.
- **Low context precision** → the retriever returns noise.
- **Low answer relevance** → the answer is fluent and grounded, but answers a different question.

Check which of these patterns appear in **your** tables, and which ones are less clear than expected. The per-question table (5.1) also shows whether one question is responsible for a change in the average.

### Advantages
- Shows **exactly where** to focus optimisation effort: retrieval or generation.
- Catches failures that are hard to see by reading answers: hallucination, noise, missing information, drift.
- Runs automatically on many questions, instead of reading every answer by hand.

### Disadvantages (our additions, not in the notes)
- Every metric needs **LLM calls** (cost and time), and the judge is a bigger model.
- The scores depend on the **judge LLM**, which can be wrong or vary a little between runs. That is why we print the judge's decisions.
- Context precision and context recall need **ground truth**, which someone has to write.

### When to use it
- Comparing two versions of a RAG system (different chunking, retriever or prompt).
- Finding out why a RAG system gives bad answers before changing anything.